<a href="https://colab.research.google.com/github/anandharidas/KernelProgramming/blob/main/matmul_Triton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Detailed Explanation & Operation Diagram of Triton GEMM

### 1. High-Level Concept
Imagine you are building a giant mosaic picture. Doing it pixel-by-pixel is slow. Instead, you divide the image into $64 \times 64$ blocks. You assign a team of workers where each worker computes exactly one block of the final result.
* Each worker finds their location in the grid (using their Program ID).
* They slide a window horizontally across Matrix $A$ and vertically down Matrix $B$, multiplying smaller $64 \times 64$ tiles and summing them up.
* Finally, they write their computed tile to Matrix $C$.

### 2. Operation Diagram
```text
                  Matrix B (N x K)
               +-----------------------+
               |                       |
               |     [b_block]         |
               |      (K x K)          |
               |                       |
               +-----------------------+
                         |  
                         |  (Matrix multiplication of blocks)
                         v  
    Matrix A             Matrix C (M x K)
    (M x N)    +-----------------------+
  +---------+  |                       |
  |         |  |                       |
  |[a_block]|->|     [accumulator]     |
  | (M x K) |  |        (Tile)         |
  +---------+  +-----------------------+
```

---

### 3. Detailed Line-by-Line Breakdown

#### Triton GPU Kernel (`gemm`)
* **`@triton.jit`**
  Compiles the Python function into highly optimized GPU machine instructions.
* **`pidx = tl.program_id(axis=0)`** & **`pidy = tl.program_id(axis=1)`**
  Identifies the coordinate of the current worker (block) in our 2D grid of execution.
* **`block_start_x`** & **`block_start_y`**
  Computes the starting index of the rows and columns for this worker's tile.
* **`offsets_x`** & **`offsets_y`**
  Creates the actual indices covering the current block span (e.g., $[128, 129, \dots, 191]$).
* **`accumulator`**
  Initializes a $64 \times 64$ grid of zeros to store the accumulating sums of our block multiplications.
* **`for i in range(0, n, BLOCK_SIZE):`**
  Loops over the internal dimension $N$ in steps of `BLOCK_SIZE` to fetch block fragments from both input matrices.
* **`a_block`** & **`b_block`**
  Loads a $64 \times 64$ block of memory from global memory into fast on-chip SRAM, masking indices to avoid out-of-bounds array reads.
* **`accumulator += tl.dot(a_block, b_block)`**
  Executes high-performance tensor-core hardware matrix multiplication on the local tiles.
* **`tl.store(...)`**
  Writes the completed $64 \times 64$ results from SRAM back to global memory in Matrix $C$.

In [19]:
import torch

In [20]:
import triton

In [21]:
import triton.language as tl

In [22]:
DEVICE = triton.runtime.driver.active.get_active_torch_device()
DEVICE

device(type='cuda', index=0)

In [23]:
@triton.jit
def gemm(
    a_ptr, b_ptr, c_ptr,
    m, n, k,
    BLOCK_SIZE: tl.constexpr
):
    pidx = tl.program_id(axis=0)
    pidy = tl.program_id(axis=1)

    block_start_x = pidx * BLOCK_SIZE
    block_start_y = pidy * BLOCK_SIZE
    offsets_x = block_start_x + tl.arange(0, BLOCK_SIZE)
    offsets_y = block_start_y + tl.arange(0, BLOCK_SIZE)

    accumulator = tl.zeros((BLOCK_SIZE, BLOCK_SIZE), dtype=tl.float32)

    for i in range(0, n, BLOCK_SIZE):
        n_offsets = i + tl.arange(0, BLOCK_SIZE)
        a_block = tl.load(
            a_ptr + offsets_x[:, None] * n + n_offsets[None, :],
            mask=(offsets_x[:, None] < m) & (n_offsets[None, :] < n),
            other=0.0
        )
        b_block = tl.load(
            b_ptr + n_offsets[:, None] * k + offsets_y[None, :],
            mask=(n_offsets[:, None] < n) & (offsets_y[None, :] < k),
            other=0.0
        )

        accumulator += tl.dot(a_block, b_block)

    tl.store(
        c_ptr + offsets_x[:, None] * k + offsets_y[None, :],
        accumulator,
        mask=(offsets_x[:, None] < m) & (offsets_y[None, :] < k)
    )


def main():
    torch.manual_seed(0)
    m = 1024
    n = 800
    k = 1000
    BLOCK_SIZE = 64
    x = torch.rand((m, n), device=DEVICE)
    y = torch.rand((n, k), device=DEVICE)
    prod = torch.zeros((m, k), device=DEVICE)

    grid = (
        triton.cdiv(m, BLOCK_SIZE),
        triton.cdiv(k, BLOCK_SIZE),
    )

    gemm[grid](x, y, prod, m, n, k, BLOCK_SIZE=BLOCK_SIZE)

    diff = (prod - x @ y).abs()
    print("max abs diff:", diff.max().item())
    print("mean abs diff:", diff.mean().item())
    print(torch.allclose(prod, x @ y, atol=1e-2, rtol=1e-2))

main()

max abs diff: 0.156036376953125
mean abs diff: 0.1323244571685791
True


### 4. Demystifying 2D-to-1D Pointer Calculations

At a high level, these lines are calculating the **1D memory addresses** of a 2D grid of elements so the GPU can load them into fast on-chip SRAM.

---

#### The Core Problem: 2D Matrices on a 1D Memory Strip

Computers store memory in a single, flat, continuous line (1D array). Even if a matrix has rows and columns (2D), in computer memory, it is flattened row-by-row:

```text
2D Grid Representation:
Row 0: [ A,  B,  C ]
Row 1: [ D,  E,  F ]

Equivalent Flat 1D Memory Layout:
Memory Index:  0    1    2    3    4    5
Element:      [A]  [B]  [C]  [D]  [E]  [F]
```
To find any element at row `r` and column `c` in a matrix with `W` columns (width), we use the flat indexing formula:
$$\text{Flat Index} = \text{Row} \times \text{Matrix Width} + \text{Column}$$

For example, to access element `E` (Row 1, Column 1, Matrix Width = 3):
$$\text{Flat Index} = 1 \times 3 + 1 = 4$$

---

#### What `offsets_x[:, None]` and `n_offsets[None, :]` Do (Broadcasting)

Triton operates on entire blocks (arrays) at once. We don't just want one element; we want a **2D grid of offsets** ($64 \times 64$ entries) in a single operation.

* **`offsets_x`** is a 1D array of row indices: `[r0, r1, r2, ...]` (shape: `[64]`)
* **`n_offsets`** is a 1D array of column indices: `[c0, c1, c2, ...]` (shape: `[64]`)

To apply our 2D formula, we use NumPy-style **broadcasting** to expand these 1D arrays into 2D coordinates:
* **`offsets_x[:, None]`**: Changes the shape from `[64]` to `[64, 1]` (a column vector of row indices).
* **`n_offsets[None, :]`**: Changes the shape from `[64]` to `[1, 64]` (a row vector of column indices).

When we combine them using addition or multiplication, Triton automatically matches (broadcasts) them to produce a $64 \times 64$ grid of calculations:

```text
    offsets_x[:, None]                n_offsets[None, :]
       (Shape: 64x1)                     (Shape: 1x64)
       
         [ r0 ]                            [ c0, c1, c2, ... ]
         [ r1 ]             +
         [ r2 ]
         [ .. ]
         
                                ||
                                \/
                        Result (Shape: 64x64)
         [ (r0, c0)  (r0, c1)  (r0, c2)  ... ]
         [ (r1, c0)  (r1, c1)  (r1, c2)  ... ]
         [ (r2, c0)  (r2, c1)  (r2, c2)  ... ]
```

---

#### Deep Dive into Matrix A & Matrix B Offsets

##### A) For Matrix A:
```python
a_ptr + offsets_x[:, None] * n + n_offsets[None, :]
```
* **`offsets_x[:, None]`**: The 2D list of target row indices ($64 \times 1$).
* **`* n`**: Multiplies each row index by the width of Matrix A ($N$). This jumps our pointer row-by-row down the matrix memory.
* **`+ n_offsets[None, :]`**: Adds the horizontal column step offsets ($1 \times 64$).
* **`a_ptr + ...`**: Adds the physical starting address of Matrix A in memory.

*This returns a $64 \times 64$ grid of exact memory addresses where our desired tile of Matrix A resides.*

##### B) For Matrix B:
```python
b_ptr + n_offsets[:, None] * k + offsets_y[None, :]
```
* Note the swap of indices! Matrix B is dimensioned $N \times K$ ($N$ rows, $K$ columns).
* **`n_offsets[:, None]`**: Represents our slice along the vertical $N$ dimension (which acts as row coordinates for Matrix B) of shape $64 \times 1$.
* **`* k`**: Multiplies by the width of Matrix B ($K$) to step row-by-row down B's memory layout.
* **`+ offsets_y[None, :]`**: Adds the $K$-dimension column indices of shape $1 \times 64$.
* **`b_ptr + ...`**: Adds the physical starting address of Matrix B in memory.

*This returns a $64 \times 64$ grid of exact memory addresses where our desired tile of Matrix B resides.*